In [ ]:
# Lecture 16 — Core Transformations: select, filter, groupBy, agg, join
# Spark 101 · Module 5 · Lecture 16
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- Two DataFrames to work with: orders and customers ---------------------
orders = spark.createDataFrame(
    [
        (1, 101, "Madrid",   19.99),
        (2, 101, "Madrid",   250.00),
        (3, 102, "Helsinki", 5.50),
        (4, 103, "New York", 1200.00),
        (5, 102, "Helsinki", 42.00),
    ],
    ["order_id", "customer_id", "city", "price"],
)

customers = spark.createDataFrame(
    [
        (101, "Ada"),
        (102, "Linus"),
        (103, "Grace"),
        (999, "Unseen"),   # a customer with no orders — matters for join type
    ],
    ["customer_id", "name"],
)

orders.show()
customers.show()

In [ ]:
# --- select — pick columns (NARROW: nothing crosses the network) -----------
orders.select("order_id", "price").show()

In [ ]:
# --- filter / where — keep matching rows (NARROW) --------------------------
from pyspark.sql.functions import col

orders.filter(col("price") > 100).show()
# `where` is an exact alias:
orders.where(col("price") > 100).show()

In [ ]:
# --- groupBy + agg — collapse to summaries (WIDE: this shuffles) -----------
from pyspark.sql.functions import sum as _sum, count, avg

orders.groupBy("city").agg(
    count("*").alias("n_orders"),
    _sum("price").alias("total"),
    avg("price").alias("avg_price"),
).show()

# shorthand for the common case:
orders.groupBy("city").count().show()

In [ ]:
# --- join — combine two DataFrames on a key (WIDE: this shuffles) ----------
# inner: only customers who have orders (999 "Unseen" drops out)
orders.join(customers, on="customer_id", how="inner") \
      .select("order_id", "name", "city", "price") \
      .show()

In [ ]:
# --- join type matters: left keeps unmatched left rows; ------------------
# right/outer surface customers with no orders (999 appears with nulls)
customers.join(orders, on="customer_id", how="left").show()

In [ ]:
# --- Put it together: filter -> groupBy -> join, then one action ----------
from pyspark.sql.functions import sum as _sum

city_totals = (
    orders
    .filter(col("price") > 10)              # narrow
    .groupBy("customer_id")                 # wide (shuffle)
    .agg(_sum("price").alias("spent"))
    .join(customers, on="customer_id")      # wide (shuffle)
    .select("name", "spent")
)
city_totals.show()

In [ ]:
# --- See the shuffles in the plan: each WIDE op is an "Exchange" -----------
city_totals.explain()
# Look for `Exchange` nodes — one per shuffle. Two wide ops -> two Exchanges.

In [ ]:
# withColumn: a computed column (price plus 10% tax).
from pyspark.sql.functions import col, round as _round
orders.withColumn("with_tax", _round(col("price") * 1.1, 2)).show()

In [ ]:
# groupBy multiple keys + multiple aggregations.
from pyspark.sql.functions import sum as _sum, avg, count
(orders.groupBy("city", "customer_id")
       .agg(count("*").alias("n"), _sum("price").alias("total"), avg("price").alias("avg"))
       .orderBy("city").show())

In [ ]:
# Join types: left keeps unmatched left rows; outer keeps everything.
orders.join(customers, "customer_id", "left").show()
orders.join(customers, "customer_id", "outer").show()

In [ ]:
# distinct, orderBy, limit — trimming and ordering results.
orders.select("city").distinct().orderBy("city").show()
orders.orderBy(col("price").desc()).limit(3).show()

In [ ]:
# Recap:
#   - select / filter are NARROW — data stays put, cheap.
#   - groupBy+agg and join are WIDE — data shuffles across the network.
#   - Chain transformations, end with an action; know every shuffle you cause.
# Next lecture: the same operations written as SQL.